# SPH Evaluation Kernels

This tutorial shows how Struphy reconstructs smooth fields from SPH particles, and verifies the reconstruction against exact solutions.

The markers live in the logical unit cube, $\boldsymbol\eta_j \in [0,1]^3$, and are mapped to physical space by the mapping $F$, $\mathbf x_j = F(\boldsymbol\eta_j)$. The smoothing kernel $W_h$ is **radial and normalized in physical space**, $\int W_h(\mathbf x)\,\textnormal d\mathbf x = 1$. The SPH density at a physical point $\mathbf x$ is

$$
\rho_h(\mathbf x) = \sum_{j=1}^{N} w_j\, W_h(\mathbf x - \mathbf x_j)\,,
$$

with the marker weights (masses) $w_j$. It is evaluated at logical points through the pull-back as a **0-form**,

$$
\hat\rho_h(\boldsymbol\eta) = \rho_h(F(\boldsymbol\eta)) = \sum_{j=1}^{N} w_j\, W_h\big(F(\boldsymbol\eta) - F(\boldsymbol\eta_j)\big)\,,
$$

so `eval_density` returns the physical density, on any mapping. The gradient is the physical gradient,

$$
\nabla \rho_h(\mathbf x) = \sum_{j=1}^{N} w_j\, \nabla W_h(\mathbf x - \mathbf x_j)\,.
$$

**What this tutorial covers**

0. The three radial kernels and their normalization.
1. The kernel width $h$: chosen at setup such that the neighbour search over the 27 sorting boxes around a point is exact.
2. 1-D reconstruction on a `Cuboid`: value, derivative, boundary conditions, tesselation vs. Monte-Carlo loading.
3. 2-D reconstruction on the curved `Colella` mapping.
4. Polar mappings: why the kernel needs an annulus.

The evaluation is done by `ParticlesSPH.eval_density`, which calls the box-based kernels in `struphy.pic.sph_eval_kernels` (cost $\mathcal O(1)$ per point).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from struphy import (
    BoundaryParameters,
    LoadingParameters,
    SortingParameters,
    SPHParameters,
    domains,
    perturbations,
)
from struphy.fields_background.equils import ConstantVelocity
from struphy.pic import sph_radial_kernels
from struphy.pic.particles import ParticlesSPH
from struphy.pic.sph_setup import KERNEL_IDS

## Part 0 — The radial kernels

All kernels have the form

$$
W_h(\mathbf r) = \frac{\sigma_d}{h^d}\, w\!\left(\frac{|\mathbf r|}{h}\right)\,, \qquad w(q) = 0 \ \textnormal{for}\ q \geq 1\,,
$$

with compact support of radius $h$, the number $d$ of resolved directions (1, 2 or 3) and the normalization constant $\sigma_d$. Three shapes are available (`SPHParameters(kernel=...)`):

| `kernel` | $w(q)$ for $q < 1$ |
|---|---|
| `"gaussian"` | $e^{-9q^2}$ (truncated at $q = 1$, renormalized) |
| `"wendland_c2"` | $(1-q)^3(1+3q)$ for $d = 1$, $(1-q)^4(1+4q)$ for $d = 2, 3$ |
| `"cubic_spline"` | M4 spline: $1 - 6q^2 + 6q^3$ for $q \leq \frac12$, $2(1-q)^3$ for $q > \frac12$ |

The Pyccel functions `sph_radial_kernels.kernel_value(kernel_id, dim, r, h)` and `kernel_dr(kernel_id, dim, r, h)` (the radial derivative) evaluate them; the ids are in `struphy.pic.sph_setup.KERNEL_IDS`.

In [ ]:
print("kernel ids:", KERNEL_IDS)

r = np.linspace(0.0, 1.0, 201)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, kid in KERNEL_IDS.items():
    w = [sph_radial_kernels.kernel_value(kid, 1, ri, 1.0) for ri in r]
    dw = [sph_radial_kernels.kernel_dr(kid, 1, ri, 1.0) for ri in r]
    axes[0].plot(r, w, label=name)
    axes[1].plot(r, dw, label=name)
axes[0].set_title(r"$W_h(r)$, $d = 1$, $h = 1$")
axes[1].set_title(r"$\partial_r W_h(r)$, $d = 1$, $h = 1$")
for ax in axes:
    ax.set_xlabel("$r$")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# normalization: int W_h dx = 1 in d = 1, 2, 3 (radial quadrature)
r = np.linspace(0.0, 1.0, 20001)
surface = {1: 2.0, 2: 2 * np.pi * r, 3: 4 * np.pi * r**2}
for name, kid in KERNEL_IDS.items():
    integrals = []
    for d in (1, 2, 3):
        w = np.array([sph_radial_kernels.kernel_value(kid, d, ri, 1.0) for ri in r])
        integrals += [np.trapezoid(w * surface[d], r)]
    print(f"{name:13s} int W_h dx in d = 1, 2, 3: " + ", ".join(f"{v:.6f}" for v in integrals))

## Part 1 — 1-D reconstruction on a Cuboid

### The kernel width

Sorting boxes partition the logical cube; the SPH sum for a point runs over the markers in the 27 boxes around it. This is exact if the kernel support (a ball of radius $h$ in physical space) is contained in these boxes. A physical displacement $\Delta\mathbf x$ changes $\eta_i$ by at most $\|(DF^{-1})_{i,:}\|\,|\Delta\mathbf x|$, which must not exceed the box width $1/n_i$. Hence the largest admissible width is

$$
h_\max = 0.99\,\min_{i\ \textnormal{resolved}} \frac{1}{n_i\, \max_{\boldsymbol\eta} \|(DF^{-1})_{i,:}(\boldsymbol\eta)\|}\,,
$$

computed at setup on a sampling grid. By default $h = h_\max$ (`SPHParameters(kernel_width=None)`). A kernel narrower than a few marker spacings gives noisy sums, so the width must also satisfy the lower bound

$$
h \geq h_\min = \texttt{min\_width\_spacings} \cdot \Big(\frac{V}{N}\Big)^{1/d}\,,
$$

with the mean marker spacing $(V/N)^{1/d}$ (default `min_width_spacings=4.0`). If $h_\max < h_\min$, the setup fails with an error message that suggests fewer sorting boxes or more markers.

Directions with a single sorting box are not resolved by the kernel: with `boxes_per_dim=(24, 1, 1)` the kernel is 1-D (in $x$), and the sum is divided by the physical cross-section $A_\perp$ of the unresolved directions, so that $\rho_h$ is still a volume density.

### Problem setup

The background density is given as a volume form in logical space, $n^3(\boldsymbol\eta) = 1.5 + \cos(2\pi\eta_1)$, so the exact physical density is $\rho = n^3 / \sqrt g$ with the Jacobian determinant $\sqrt g$ of the mapping. We use a **tesselation** loading (markers on a regular lattice), so the error is controlled by the lattice spacing rather than Monte-Carlo noise.

In [ ]:
# Cuboid domain with Lx = 2: x in [0, 2], cross-section 10 x 10
domain_1d = domains.Cuboid(l1=0.0, r1=2.0, l2=0.0, r2=10.0, l3=0.0, r3=10.0)
Lx = 2.0
sqrt_g = 2.0 * 10.0 * 10.0

background_1d = ConstantVelocity(n=1.5, density_profile="constant")
background_1d.domain = domain_1d
pert_1d = {"n": perturbations.ModesCos(ls=(1,), amps=(1.0,))}

# exact physical density and its x-derivative (x = Lx * eta1)
rho_exact = lambda e1: (1.5 + np.cos(2 * np.pi * e1)) / sqrt_g
drho_exact = lambda e1: -2 * np.pi / Lx * np.sin(2 * np.pi * e1) / sqrt_g

boxes_per_dim = (24, 1, 1)
sorting_params = SortingParameters(boxes_per_dim=boxes_per_dim)

n_eval = 200
ee1, ee2, ee3 = np.meshgrid(np.linspace(0, 1, n_eval), [0.5], [0.5], indexing="ij")
x_plot = Lx * ee1.squeeze()


def make_particles_1d(bc_x="periodic", ppb=5, loading="tesselation", kernel="gaussian", kernel_width=None):
    # Construct and initialise a 1-D ParticlesSPH object.
    particles = ParticlesSPH(
        comm_world=None,
        loading_params=LoadingParameters(ppb=ppb, seed=1607, loading=loading),
        boundary_params=BoundaryParameters(bc_sph=(bc_x, "periodic", "periodic")),
        sorting_params=sorting_params,
        sph_params=SPHParameters(kernel=kernel, kernel_width=kernel_width),
        bufsize=1.0,
        domain=domain_1d,
        background=background_1d,
        perturbations=pert_1d,
        n_as_volume_form=True,
    )
    particles.draw_markers(sort=False)
    particles.initialize_weights()
    return particles


p = make_particles_1d()
k = p.sph_kernel
print(f"box width in x: {Lx / boxes_per_dim[0]:.4f}")
print(f"h = {k.h:.4f}  (h_max = {k.h_max:.4f}, h_min = {k.h_min:.4f}), resolved dims: {k.dim}, 1/A_perp = {k.scale}")

### 1.1 Density value, three kernels (periodic)

In [ ]:
kernels = list(KERNEL_IDS)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle(r"1-D density reconstruction  ($\rho = (1.5 + \cos(2\pi x/L_x))/\sqrt{g}$, periodic BC)")

for ax, kernel in zip(axes, kernels):
    p = make_particles_1d(kernel=kernel)
    rho_sph = p.eval_density(ee1, ee2, ee3).squeeze()
    rho_ex = rho_exact(ee1.squeeze())
    err = np.max(np.abs(rho_sph - rho_ex)) / np.max(np.abs(rho_ex))
    ax.plot(x_plot, rho_ex, "k-", lw=2, label="exact")
    ax.plot(x_plot, rho_sph, "r--", lw=1.5, label=f"SPH  (err={err:.2e})")
    ax.set_title(kernel)
    ax.set_xlabel("$x$")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel(r"$\rho$")
plt.tight_layout()
plt.show()

### 1.2 Derivative

`derivative=1` returns $\partial\rho_h/\partial x$, the first component of the **physical** gradient (here $\partial_x = L_x^{-1}\partial_{\eta_1}$). Derivatives amplify the lattice error, so we use more markers per box.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle(r"1-D derivative reconstruction  ($\partial_x \rho$, periodic BC)")

for ax, kernel in zip(axes, kernels):
    p = make_particles_1d(ppb=32, kernel=kernel)
    drho_sph = p.eval_density(ee1, ee2, ee3, derivative=1).squeeze()
    drho_ex = drho_exact(ee1.squeeze())
    err = np.max(np.abs(drho_sph - drho_ex)) / np.max(np.abs(drho_ex))
    ax.plot(x_plot, drho_ex, "k-", lw=2, label="exact")
    ax.plot(x_plot, drho_sph, "r--", lw=1.5, label=f"SPH  (err={err:.2e})")
    ax.set_title(kernel)
    ax.set_xlabel("$x$")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel(r"$\partial_x\rho$")
plt.tight_layout()
plt.show()

### 1.3 Boundary conditions

Struphy supports these SPH boundary conditions (`BoundaryParameters(bc_sph=...)`):

| BC | Description |
|---|---|
| `"periodic"` | Markers near the opposite boundary contribute (shifted by the period vector). |
| `"mirror"` | Ghost particles reflected at the wall; zero normal gradient. |
| `"fixed"` | Ghost particles reflected, with weights from the initial condition at the mirrored position. |
| `"noslip"` | Like `"mirror"`, with reflected velocities (for viscous flows). |

The interior is the same for all of them; they differ only within one kernel width of the wall.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_plot, rho_exact(ee1.squeeze()), "k-", lw=1.5, label="exact", zorder=5)
for bc, color in zip(["periodic", "mirror", "fixed"], ["tab:blue", "tab:orange", "tab:green"]):
    p = make_particles_1d(bc_x=bc)
    ax.plot(x_plot, p.eval_density(ee1, ee2, ee3).squeeze(), "--", color=color, lw=2.5, label=f"bc={bc!r}")
ax.set_title("Boundary condition comparison (gaussian)")
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\rho$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1.4 Tesselation vs Monte-Carlo loading

With **tesselation** loading the markers sit on a regular lattice (`ppb` per sorting box). With **pseudo-random** loading the positions are random, and the sampling noise decays only as $\mathcal O(N^{-1/2})$: ten times more markers still give visible fluctuations.

In [ ]:
ppb_tess = 5
ppb_rand = 10 * ppb_tess
p_tess = make_particles_1d(ppb=ppb_tess, loading="tesselation")
p_rand = make_particles_1d(ppb=ppb_rand, loading="pseudo_random")

rho_ex = rho_exact(ee1.squeeze())
rho_tess = p_tess.eval_density(ee1, ee2, ee3).squeeze()
rho_rand = p_rand.eval_density(ee1, ee2, ee3).squeeze()
err_tess = np.max(np.abs(rho_tess - rho_ex)) / np.max(np.abs(rho_ex))
err_rand = np.max(np.abs(rho_rand - rho_ex)) / np.max(np.abs(rho_ex))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_plot, rho_ex, "k-", lw=2, label="exact")
ax.plot(x_plot, rho_tess, "b-", lw=1.5, label=f"tesselation  Np={p_tess.Np}  err={err_tess:.2e}")
ax.plot(x_plot, rho_rand, "r-", lw=1, label=f"pseudo_random  Np={p_rand.Np}  err={err_rand:.2e}")
ax.set_title("Tesselation vs Monte Carlo (gaussian, periodic)")
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\rho$")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 1.5 Choosing the width

A smaller width can be set with `SPHParameters(kernel_width=h)` (in physical units), as long as $h_\min \leq h \leq h_\max$. Outside this range the setup raises an error that explains the limits:

In [ ]:
p = make_particles_1d(ppb=10, kernel_width=0.06)
print(f"h = {p.sph_kernel.h}")

try:
    make_particles_1d(ppb=10, kernel_width=0.02)
except ValueError as e:
    print("ValueError:", e)

## Part 2 — 2-D reconstruction on a curved mapping

On the `Colella` mapping

$$
x = L_x\big(\eta_1 + \alpha \sin(2\pi\eta_1)\sin(2\pi\eta_2)\big)\,,\qquad y = L_y\big(\eta_2 + \alpha \sin(2\pi\eta_1)\sin(2\pi\eta_2)\big)
$$

the Jacobian determinant varies in space. With the volume form $n^3 = 1.5 + \cos(2\pi\eta_1)\cos(2\pi\eta_2)$ the exact physical density is $\rho = n^3/\sqrt g$; the SPH sum in physical space reproduces it directly, without any metric correction. With $12 \times 12$ sorting boxes the kernel is 2-D (in $x, y$); $h_\max$ is now limited by the most stretched cell of the mapping.

In [ ]:
domain_2d = domains.Colella(Lx=1.0, Ly=1.0, alpha=0.05, Lz=1.0)

background_2d = ConstantVelocity(n=1.5, density_profile="constant")
background_2d.domain = domain_2d
pert_2d = {"n": perturbations.ModesCosCos(ls=(1,), ms=(1,), amps=(1.0,))}

n3_2d = lambda e1, e2: 1.5 + np.cos(2 * np.pi * e1) * np.cos(2 * np.pi * e2)

boxes_per_dim_2d = (12, 12, 1)


def make_particles_2d(ppb=49, kernel="gaussian"):
    particles = ParticlesSPH(
        comm_world=None,
        loading_params=LoadingParameters(ppb=ppb, loading="tesselation"),
        boundary_params=BoundaryParameters(bc_sph=("periodic", "periodic", "periodic")),
        sorting_params=SortingParameters(boxes_per_dim=boxes_per_dim_2d),
        sph_params=SPHParameters(kernel=kernel),
        bufsize=1.0,
        domain=domain_2d,
        background=background_2d,
        perturbations=pert_2d,
        n_as_volume_form=True,
    )
    particles.draw_markers(sort=False)
    particles.initialize_weights()
    return particles


n_eval_2d = 50
ee1_2d, ee2_2d, ee3_2d = np.meshgrid(np.linspace(0, 1, n_eval_2d), np.linspace(0, 1, n_eval_2d), [0.0], indexing="ij")

# physical coordinates and exact density
X, Y, _ = domain_2d(ee1_2d, ee2_2d, ee3_2d)
X, Y = X.squeeze(), Y.squeeze()
sg = domain_2d.jacobian_det(ee1_2d, ee2_2d, ee3_2d).squeeze()
rho_ex_2d = n3_2d(ee1_2d.squeeze(), ee2_2d.squeeze()) / sg

particles_2d = make_particles_2d()
k = particles_2d.sph_kernel
print(f"Np = {particles_2d.Np}, h = {k.h:.4f} (h_max = {k.h_max:.4f}, h_min = {k.h_min:.4f}), resolved dims: {k.dim}")

### 2.1 Density value

In [ ]:
rho_sph_2d = particles_2d.eval_density(ee1_2d, ee2_2d, ee3_2d).squeeze()
err_2d = np.max(np.abs(rho_sph_2d - rho_ex_2d)) / np.max(np.abs(rho_ex_2d))
print(f"Max relative error: {err_2d:.3e}")

vmin, vmax = rho_ex_2d.min(), rho_ex_2d.max()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle("2-D density reconstruction on Colella (gaussian, periodic)")
im0 = axes[0].pcolormesh(X, Y, rho_ex_2d, vmin=vmin, vmax=vmax, shading="gouraud")
axes[0].set_title(r"Exact $n^3/\sqrt{g}$")
fig.colorbar(im0, ax=axes[0])
im1 = axes[1].pcolormesh(X, Y, rho_sph_2d, vmin=vmin, vmax=vmax, shading="gouraud")
axes[1].set_title("SPH")
fig.colorbar(im1, ax=axes[1])
im2 = axes[2].pcolormesh(X, Y, np.abs(rho_sph_2d - rho_ex_2d), shading="gouraud", cmap="Reds")
axes[2].set_title(f"Point-wise error  (max={err_2d:.2e})")
fig.colorbar(im2, ax=axes[2])
for ax in axes:
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### 2.2 Physical gradient

`derivative=1` and `derivative=2` give $\partial_x\rho_h$ and $\partial_y\rho_h$. The exact gradient follows from the chain rule, $\nabla_{\mathbf x}\rho = DF^{-\top}\nabla_{\boldsymbol\eta}\rho$, here with finite differences in $\boldsymbol\eta$.

In [ ]:
particles_2d_deriv = make_particles_2d(ppb=64)

eps = 1e-6
rho_fun = lambda e1, e2: (n3_2d(e1, e2) / domain_2d.jacobian_det(e1, e2, ee3_2d)).squeeze()
g_eta = [
    (rho_fun(ee1_2d + eps, ee2_2d) - rho_fun(ee1_2d - eps, ee2_2d)) / (2 * eps),
    (rho_fun(ee1_2d, ee2_2d + eps) - rho_fun(ee1_2d, ee2_2d - eps)) / (2 * eps),
]
dfinv = domain_2d.jacobian_inv(ee1_2d, ee2_2d, ee3_2d).squeeze()  # dfinv[i, k] = d eta_i / d x_k

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("2-D gradient reconstruction on Colella (gaussian, ppb=64)")
for row, (der, label) in enumerate([(1, r"$\partial_x\rho$"), (2, r"$\partial_y\rho$")]):
    d_ex = dfinv[0, der - 1] * g_eta[0] + dfinv[1, der - 1] * g_eta[1]
    d_sph = particles_2d_deriv.eval_density(ee1_2d, ee2_2d, ee3_2d, derivative=der).squeeze()
    err = np.max(np.abs(d_sph - d_ex)) / np.max(np.abs(d_ex))
    vmin_d, vmax_d = d_ex.min(), d_ex.max()
    im0 = axes[row, 0].pcolormesh(X, Y, d_ex, vmin=vmin_d, vmax=vmax_d, shading="gouraud")
    axes[row, 0].set_title(f"Exact {label}")
    fig.colorbar(im0, ax=axes[row, 0])
    im1 = axes[row, 1].pcolormesh(X, Y, d_sph, vmin=vmin_d, vmax=vmax_d, shading="gouraud")
    axes[row, 1].set_title(f"SPH {label}")
    fig.colorbar(im1, ax=axes[row, 1])
    im2 = axes[row, 2].pcolormesh(X, Y, np.abs(d_sph - d_ex), shading="gouraud", cmap="Reds")
    axes[row, 2].set_title(f"Error  (max={err:.2e})")
    fig.colorbar(im2, ax=axes[row, 2])
for ax in axes.flat:
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### 2.3 Kernel comparison in 2-D

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle("Kernel comparison on Colella: point-wise error of the density")
for ax, kernel in zip(axes, kernels):
    p = make_particles_2d(kernel=kernel)
    rho_sph = p.eval_density(ee1_2d, ee2_2d, ee3_2d).squeeze()
    err = np.max(np.abs(rho_sph - rho_ex_2d)) / np.max(np.abs(rho_ex_2d))
    im = ax.pcolormesh(X, Y, np.abs(rho_sph - rho_ex_2d), shading="gouraud", cmap="Reds")
    ax.set_title(f"{kernel}\nerr={err:.2e}")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_aspect("equal")
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Part 3 — Polar mappings need an annulus

On a polar mapping (e.g. `HollowCylinder`, with $\eta_1$ radial and $\eta_2$ angular), the angular width of a sorting box shrinks to zero at the axis, $2\pi r / n_2$, and so does $h_\max$. The kernel width is limited by the inner radius $a_1$: the setup fails if $h_\max < h_\min$, and the error message suggests an annulus with a larger inner radius (or fewer angular boxes).

In [ ]:
def make_polar(a1, boxes=(8, 16, 1)):
    bg = ConstantVelocity(n=1.0, density_profile="constant")
    domain = domains.HollowCylinder(a1=a1, a2=1.0, Lz=1.0)
    bg.domain = domain
    p = ParticlesSPH(
        comm_world=None,
        loading_params=LoadingParameters(ppb=64, loading="tesselation"),
        boundary_params=BoundaryParameters(bc_sph=("mirror", "periodic", "periodic")),
        sorting_params=SortingParameters(boxes_per_dim=boxes),
        bufsize=1.0,
        domain=domain,
        background=bg,
        n_as_volume_form=True,
    )
    return p


for a1 in (0.1, 0.3):
    try:
        k = make_polar(a1).sph_kernel
        print(f"a1 = {a1}: h = {k.h:.4f} (h_max = {k.h_max:.4f}, h_min = {k.h_min:.4f})")
    except ValueError as e:
        print(f"a1 = {a1}: ValueError: {e}\n")

## Summary

| Task | How |
|---|---|
| Physical density $\rho_h(F(\boldsymbol\eta))$ | `particles.eval_density(eta1, eta2, eta3)` |
| Physical gradient $\partial_{x_k}\rho_h$ | `derivative=k` ($k = 1, 2, 3$) |
| Kernel shape | `SPHParameters(kernel="gaussian" \| "wendland_c2" \| "cubic_spline")` |
| Kernel width | `SPHParameters(kernel_width=None)` gives $h_\max$; a value in physical units must lie in $[h_\min, h_\max]$ |
| Lower bound | `SPHParameters(min_width_spacings=4.0)`, in units of the mean marker spacing |
| Resolved directions | the directions with more than one sorting box |

In a simulation the kernel is set per species, `species.set_markers(..., sph_params=SPHParameters(...))`, and shared by all propagators and diagnostics of the species. For the implementation see `struphy.pic.sph_setup`, `struphy.pic.sph_radial_kernels` and `struphy.pic.sph_eval_kernels`.